# 🎬 Produção Automática de Vídeo para YouTube
## Google Colab - Processamento em Nuvem Gratuito

**Instruções:**
1. Execute cada célula em ordem (clique no ▶️)
2. Faça upload do seu áudio quando solicitado
3. Aguarde processamento
4. Baixe vídeo e thumbnail prontos!

**Custo:** R$ 0,00 (Google Colab Free Tier)

## 📦 CÉLULA 1: Instalar Dependências

In [ ]:
!pip install moviepy pydub pillow requests -q
!apt-get install ffmpeg imagemagick -y -qq
print("✅ Dependências instaladas!")

## ⚙️ CÉLULA 2: Importar Bibliotecas

In [ ]:
from google.colab import files, drive
from IPython.display import display, Audio, Video
import os, requests, random
from PIL import Image, ImageDraw, ImageFont
from moviepy.editor import *

# Configurar pasta
WORK_DIR = "/content/youtube_production"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✅ Ambiente pronto em: {WORK_DIR}")

## 🎵 CÉLULA 3: Upload do Áudio

In [ ]:
print("📤 Upload do arquivo de áudio (MP3, WAV, OGG)")
uploaded = files.upload()

if uploaded:
    audio_path = list(uploaded.keys())[0]
    audio = AudioFileClip(audio_path)
    duracao = audio.duration
    print(f"✅ Áudio: {audio_path} | Duração: {duracao:.1f}s ({duracao/60:.1f} min)")
    display(Audio(audio_path))
else:
    raise Exception("Nenhum arquivo enviado!")

## 🎨 CÉLULA 4: Gerar Thumbnail

In [ ]:
titulo = input("Título do vídeo: ") or "Meu Vídeo YouTube"
tema = input("Tema (ex: educacao, tecnologia): ") or "educacao"

# Criar thumbnail
width, height = 1280, 720
img = Image.new('RGB', (width, height), color=(20, 20, 80))
draw = ImageDraw.Draw(img)

# Gradiente
for y in range(height):
    r = int(20 + (y/height) * 50)
    g = int(20 + (y/height) * 50)
    b = int(80 + (y/height) * 100)
    draw.line([(0, y), (width, y)], fill=(r, g, b))

# Texto
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 80)
except:
    font = ImageFont.load_default()

text = titulo.upper()[:50]
bbox = draw.textbbox((0, 0), text, font=font)
x = (width - (bbox[2] - bbox[0])) // 2
y = (height - (bbox[3] - bbox[1])) // 2

draw.text((x+4, y+4), text, fill='black', font=font)
draw.text((x, y), text, fill='white', font=font)

# Salvar
thumbnail_file = "thumbnail.png"
img.save(thumbnail_file, 'PNG')
print(f"✅ Thumbnail gerada: {thumbnail_file}")
display(Image.open(thumbnail_file))

## 🎬 CÉLULA 5: Baixar Imagens de Stock

In [ ]:
def baixar_imagem(tema, idx):
    url = f"https://source.unsplash.com/1920x1080/?{tema},{idx}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            filename = f"img_{tema}_{idx}.jpg"
            with open(filename, 'wb') as f:
                f.write(resp.content)
            return filename
    except:
        pass
    
    # Fallback
    img = Image.new('RGB', (1920, 1080), color=(random.randint(50,150), random.randint(50,150), random.randint(50,150)))
    filename = f"img_solid_{idx}.jpg"
    img.save(filename)
    return filename

# Baixar 5-10 imagens
num_cenas = max(5, int(duracao / 10))
imagens = []
for i in range(num_cenas):
    img_file = baixar_imagem(tema, i+1)
    imagens.append(img_file)
    print(f"✅ Imagem {i+1}/{num_cenas}: {img_file}")

print(f"\nTotal: {len(imagens)} imagens prontas")

## 🎞️ CÉLULA 6: Montar Vídeo

In [ ]:
print("🎬 Montando vídeo...")

# Criar clips das imagens
duracao_cena = duracao / len(imagens)
clips = []

for img_file in imagens:
    clip = ImageClip(img_file).set_duration(duracao_cena)
    clip = clip.resize(lambda t: 1 + 0.04*t)  # Zoom suave
    clips.append(clip)

# Concatenar
video = concatenate_videoclips(clips, method="compose")
video = video.set_duration(duracao)
video = video.set_audio(audio)
video = video.fadein(1).fadeout(2)

# Exportar
output_file = "video_final.mp4"
print(f"⏳ Renderizando... (3-5 minutos)")

video.write_videofile(
    output_file,
    fps=24,
    codec='libx264',
    audio_codec='aac',
    temp_audiofile='temp.m4a',
    remove_temp=True,
    preset='medium',
    bitrate='5000k'
)

tamanho_mb = os.path.getsize(output_file) / 1024 / 1024
print(f"✅ Vídeo renderizado: {output_file} ({tamanho_mb:.1f} MB)")

## 📥 CÉLULA 7: Preview e Download

In [ ]:
print("🎬 Preview:")
display(Video(output_file, width=640))

print("\n📥 Baixando arquivos...")
files.download(output_file)
files.download(thumbnail_file)

print("\n" + "="*50)
print("  ✅ PRODUÇÃO CONCLUÍDA!")
print("="*50)
print(f"\nArquivos:")
print(f"  • Vídeo: {output_file}")
print(f"  • Thumbnail: {thumbnail_file}")
print(f"\nPróximo passo: Upload no YouTube Studio!")